# LeetCode #1425: Constrained Subset Sum

https://leetcode.com/problems/constrained-subset-sum/

## Comparison of Approaches

| Approach | Time Complexity | Space Complexity |
| :--- | :--- | :--- |
| **Brute Force** | $O(nk)$ | $O(n)$ |
| **Optimal: Monotonic Deque DP ★** | $O(n)$ | $O(n)$ |

---

## Understanding the Methods

### Brute Force
For each index `i`, scan the previous `k` elements to find the maximum `dp[j]`, then set `dp[i] = nums[i] + max(0, max_dp)`. This is $O(nk)$ — too slow for large $n, k$.

### Optimal: Monotonic Deque DP ★
Maintain a monotonically decreasing deque of `dp` indices. The front always holds the index with the largest `dp` value within the last `k` positions. For each `i`, `dp[i] = nums[i] + max(0, dp[deque.front])`. Evict expired indices from the front; evict smaller values from the back before pushing `i`. Each index is pushed and popped at most once — $O(n)$ total.

**Constraints:**
* `1 <= nums.length <= 10^5`
* `-10^4 <= nums[i] <= 10^4`
* `1 <= k <= nums.length`

## Solutions

### C#

In [ ]:
public class Solution {
    public int ConstrainedSubsetSum(int[] nums, int k) {
        int n = nums.Length;
        int[] dp = new int[n];
        // Monotonically decreasing deque of dp-value indices
        var dq = new LinkedList<int>();
        int ans = int.MinValue;

        for (int i = 0; i < n; i++) {
            // Discard indices that are too far away (outside the k-window)
            while (dq.Count > 0 && i - dq.First.Value > k) dq.RemoveFirst();

            // Best previous dp value in the window (only useful if positive)
            dp[i] = nums[i] + (dq.Count > 0 ? Math.Max(0, dp[dq.First.Value]) : 0);
            ans = Math.Max(ans, dp[i]);

            // Maintain decreasing order: remove back elements smaller than dp[i]
            while (dq.Count > 0 && dp[dq.Last.Value] <= dp[i]) dq.RemoveLast();
            dq.AddLast(i);
        }
        return ans;
    }
}

### Python

In [ ]:
from collections import deque

class Solution:
    def constrainedSubsetSum(self, nums: list[int], k: int) -> int:
        n = len(nums)
        dp = [0] * n
        # Monotonically decreasing deque stores indices of dp array
        dq: deque[int] = deque()
        ans = float('-inf')

        for i in range(n):
            # Remove indices outside the k-window from the front
            while dq and i - dq[0] > k:
                dq.popleft()

            # Extend by the best reachable previous dp value (ignore negatives)
            dp[i] = nums[i] + (max(0, dp[dq[0]]) if dq else 0)
            ans = max(ans, dp[i])

            # Discard smaller dp values from the back — they can never be optimal
            while dq and dp[dq[-1]] <= dp[i]:
                dq.pop()
            dq.append(i)

        return ans

### Go

In [ ]:
func constrainedSubsetSum(nums []int, k int) int {
    n := len(nums)
    dp := make([]int, n)
    // Monotonically decreasing deque of dp indices
    dq := []int{}
    ans := -(1 << 30)

    for i := 0; i < n; i++ {
        // Expire indices beyond the k-window
        for len(dq) > 0 && i-dq[0] > k { dq = dq[1:] }

        // Best previous dp (zero-floor it so we never drag the sum negative)
        prev := 0
        if len(dq) > 0 && dp[dq[0]] > 0 { prev = dp[dq[0]] }
        dp[i] = nums[i] + prev
        if dp[i] > ans { ans = dp[i] }

        // Keep deque decreasing: evict tail entries dominated by dp[i]
        for len(dq) > 0 && dp[dq[len(dq)-1]] <= dp[i] { dq = dq[:len(dq)-1] }
        dq = append(dq, i)
    }
    return ans
}

### Rust

In [ ]:
use std::collections::VecDeque;

impl Solution {
    pub fn constrained_subset_sum(nums: Vec<i32>, k: i32) -> i32 {
        let k = k as usize;
        let n = nums.len();
        let mut dp = vec![0i32; n];
        // Monotonically decreasing deque of dp indices
        let mut dq: VecDeque<usize> = VecDeque::new();
        let mut ans = i32::MIN;

        for i in 0..n {
            // Remove expired indices from the front
            while dq.front().map_or(false, |&f| i - f > k) { dq.pop_front(); }

            // Take the best non-negative previous dp value from the deque front
            let prev = dq.front().map_or(0, |&f| dp[f].max(0));
            dp[i] = nums[i] + prev;
            ans = ans.max(dp[i]);

            // Evict smaller dp values from the back before inserting i
            while dq.back().map_or(false, |&b| dp[b] <= dp[i]) { dq.pop_back(); }
            dq.push_back(i);
        }
        ans
    }
}

## Example Scenarios

### 1. Common Case
**Input:** `nums = [10,2,-10,5,20], k = 2`
`dp = [10, 12, 2, 17, 37]`. The deque keeps the highest dp reachable within 2 steps. Answer: **37**.

### 2. Slightly Complex
**Input:** `nums = [-1,-2,-3], k = 1`
All values negative; best is to take no previous element. `dp = [-1,-2,-3]`, answer: **-1**.

### 3. Edge Case: Time Factor
**Input:** $n = 10^5$ elements, $k = 10^5$.
Each index is enqueued and dequeued at most once — amortised $O(n)$ total deque operations.

### 4. Edge Case: Space Factor
**Input:** $n = 10^5$ monotonically increasing values.
Deque may hold up to $k$ indices simultaneously — $O(k)$ extra space, bounded by $O(n)$.

### 5. Almost-Impossible but Plausible
**Input:** `nums = [10000]*100000, k = 1`
Every element can build on its neighbour; `dp[i] = 10000*(i+1)`. Max = $10000 \times 100000 = 10^9$ — safely within 32-bit signed range.